In [0]:
catalog = 'ecommerce_dev'

silver_orders_table = f"{catalog}.silver.orders"
silver_customers_table = f"{catalog}.silver.customers"
silver_products_table = f"{catalog}.silver.products"


gold_daily_sales_table = f"{catalog}.gold.daily_sales_kpis"
gold_customer_segmentation_table = f"{catalog}.gold.customer_segmentation"
gold_category_performance_table = f"{catalog}.gold.category_performance"

In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    countDistinct,
    when,
    round,
    lit
)

In [0]:
df_orders = spark.table(silver_orders_table)
df_customers = spark.table(silver_customers_table)
df_products =  spark.table(silver_products_table)

In [0]:
df_daily_sales_kpis = (
    df_orders
    .groupBy("order_date")
    .agg(
        count("order_id").alias("total_order_count"),

        sum(col("quantity") * col("unit_price")).alias("gross_revenue"),

        sum(
            when(
                col("order_status") == "COMPLETED",
                col("quantity") * col("unit_price")
            ).otherwise(lit(0))
        ).alias("net_completed_revenue"),

        sum(
            when(col("order_status") == "COMPLETED", 1).otherwise(0)
        ).alias("completed_orders_count"),

        sum(
            when(col("order_status") == "RETURNED", 1).otherwise(0)
        ).alias("returned_orders_count"),

        sum(
            when(col("order_status") == "CANCELLED", 1).otherwise(0)
        ).alias("cancelled_orders_count")
    )
)

In [0]:
df_daily_sales_kpis = (
    df_daily_sales_kpis
    .withColumn(
        "return_rate_percentage",
        round(
            col("cancelled_orders_count") / col("total_order_count") * 100, 2
        )
    )
    .orderBy("order_date")
)

In [0]:
display(df_daily_sales_kpis)

In [0]:
(
    df_daily_sales_kpis.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_daily_sales_table)
)

In [0]:
df_daily_sales_check = spark.table("ecommerce_dev.gold.daily_sales_kpis")

display(df_daily_sales_check)

In [0]:
df_completed_orders = df_orders.filter(
    col("order_status") == "COMPLETED"
)

In [0]:
df_customer_orders = df_completed_orders.join(
    df_customers,
    on="customer_id",
    how="inner"
)

In [0]:
df_customer_segmentation = (
    df_customer_orders
    .groupBy("customer_id", "customer_name", "city", "country")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(
            sum(col("quantity") * col("unit_price")), 2
        ).alias("lifetime_spend")
    )
    .withColumn(
        "average_order_value",
        round(col("lifetime_spend") / col("total_orders"), 2)
    )
)

In [0]:
df_customer_segmentation = (
    df_customer_segmentation
    .withColumn(
        "customer_tier",
        when(col("lifetime_spend") >= 1000, "High Value")
        .when(col("lifetime_spend") >= 300, "Medium Value")
        .otherwise("Standard")
    )
    .orderBy(col("lifetime_spend").desc())
)

In [0]:
display(df_customer_segmentation)

In [0]:
(
    df_customer_segmentation
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_customer_segmentation_table)
)

In [0]:
df_customer_segmentation_check = spark.table(
    gold_customer_segmentation_table
)

display(df_customer_segmentation_check)

In [0]:
df_completed_category_orders = df_orders.filter(
    col("order_status") == "COMPLETED"
)

In [0]:
df_category_orders = (
    df_orders.alias("o")
    .join(
        df_products.alias("p"),
        col("o.product_id") == col("p.product_id"),
        "inner"
    )

    .select(
        col("o.order_id"),
        col("o.product_id"),
        col("o.order_status"),
        col("o.quantity"),
        col("o.unit_price"),
        col("p.category")
    )
)

In [0]:
df_category_performance = (
    df_category_orders
    .groupBy("category")
    .agg(
        sum(
            when(
                col("order_status") == "COMPLETED",
                col("quantity")
            ).otherwise(0)
        ).alias("total_units_sold"),

        round(
            sum(
                when(
                    col("order_status") == "COMPLETED",
                    col("quantity") * col("unit_price")
                ).otherwise(0)
            ), 2
        ).alias("category_gross_revenue"),

        sum(
            when(col("order_status") == "RETURNED", 1).otherwise(0)
        ).alias("total_returns")
    )
)

In [0]:
display(df_category_performance)

In [0]:
(
    df_category_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_category_performance_table)
)

In [0]:
df_category_performance_check = spark.table(gold_category_performance_table)

display(df_category_performance_check)

In [0]:
gold_tables = [
    gold_daily_sales_table,
    gold_customer_segmentation_table,
    gold_category_performance_table
]

In [0]:
for table_name in gold_tables:
    df = spark.table(table_name)

    print(f"Table: {table_name}")
    print(f"Row count: {df.count()}")
    df.printSchema()